In [2]:
import pandas as pd

df2 = pd.read_json('../data/ragtruth/raw/source_info.jsonl', lines=True)
df2.to_csv("../data/ragtruth/source_dataset.csv", index=False)
df2.head(10)
print(df2['task_type'].value_counts())
df2.columns
df2.head(10)

task_type
Data2txt    1033
QA           989
Summary      943
Name: count, dtype: int64


,source_id,task_type,source,source_info,prompt
0,15592,Summary,CNN/DM,"Seventy years ago, Anne Frank died of typhus i...",Summarize the following news within 116 words:...
1,15593,Summary,CNN/DM,"Andrew Getty, one of the heirs to billions of ...",Summarize the following news within 89 words:\...
2,15594,Summary,CNN/DM,"Since Iran's Islamic Revolution in 1979, women...",Summarize the following news within 98 words:\...
3,15595,Summary,CNN/DM,"It's the kind of thing you see in movies, like...",Summarize the following news within 200 words:...
4,15596,Summary,CNN/DM,The FBI charged a Philadelphia woman on Thursd...,Summarize the following news within 86 words:\...
5,15597,Summary,CNN/DM,"Police in the Indian city of Malegaon, in the ...",Summarize the following news within 97 words:\...
6,15598,Summary,CNN/DM,Atlanta (CNN)It was a scene worthy of any top ...,Summarize the following news within 108 words:...
7,15599,Summary,CNN/DM,Blue Bell ice cream has temporarily shut down ...,Summarize the following news within 161 words:...
8,15600,Summary,CNN/DM,When Hong Kong police answered a call in the e...,Summarize the following news within 173 words:...
9,15601,Summary,CNN/DM,According to an outside review by Columbia Jou...,Summarize the following news within 200 words:...


In [3]:
df_qa = df2[df2["task_type"]=="QA"] 
print(df_qa.head(2))
print(df_qa.shape)
print(df_qa.columns)
df_qa.loc[1976, 'source_info']


      source_id task_type source  \
1976      14292        QA  MARCO   
1977      14293        QA  MARCO   

                                            source_info  \
1976  {'question': 'butcher shop phone number', 'pas...   
1977  {'question': 'tips how to conserve water', 'pa...   

                                                 prompt  
1976  Briefly answer the following question:\nbutche...  
1977  Briefly answer the following question:\ntips h...  
(989, 5)
Index(['source_id', 'task_type', 'source', 'source_info', 'prompt'], dtype='object')


{'question': 'butcher shop phone number',
 'passages': 'passage 1:Butcher Shop - Hayward 826 B Street, Hayward CA 94541 Phone Number: (510) 889-8690\n\npassage 2:Butcher Shop - Lakefield 212 Main Street, Lakefield MN 56150 Phone Number: (507) 662-6281\n\npassage 3:Some More Phone Numbers Related To The Butcher Shop The Local Butcher Shop phone number : (510) 845-6328 Boones Butcher Shop phone number : (502) 348-3668 Ye Ole Butcher Shop phone number : (972) 423-1848 J Ms Butcher Shop phone number : (865) 483-9228\n\n'}

In [4]:
from pprint import pprint
pprint(df_qa.loc[1980,"source_info"])
source_id_df2 = df_qa['source_id']


{'passages': 'passage 1:Gelish Soak Off Gel Nail Polish is a unique formula '
             'that lasts up to 21 days on natural nails and the application '
             'out performs other Soak Off Gel Polishes.Excellent coverage with '
             'quick and accurate cure times, and a easy removal process. '
             'Special designed color pigments create the ultimate in Soak Off '
             'Gel Polish.elish Soak Off Gel Nail Polish is a unique formula '
             'that lasts up to 21 days on natural nails and the application '
             'out performs other Soak Off Gel Polishes.\n'
             '\n'
             'passage 2:1 This helps break through the top coat to allow the '
             'remover to penetrate the layers of polish below. 2  Soak a '
             'cotton ball with the acetone remover and place it on one of the '
             'fingernails. 3  Wrap the finger with the cotton ball in foil, '
             'and squeeze it nice and tight so that the cotton 

In [5]:
source_info = df_qa['source_info'].to_dict()
source_info
df_ques_source = pd.DataFrame(source_info)
#print(df_ques_source.head(2))
df_ques_source = df_ques_source.transpose()
df_ques_source.head(2)
df_ques_source['source_id'] = source_id_df2
df_ques_source.reset_index(drop=True, inplace=True)
df_ques_source.head(2)
print(df_ques_source['source_id'].duplicated().sum())
df_ques_source.head(2)




0


,question,passages,source_id
0,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292
1,tips how to conserve water,passage 1:1 Take shorter showers. 2 Replace y...,14293


## response JSON

In [6]:
#df_res = pd.read_json('../data/ragtruth/raw/response.jsonl', lines=True)
#print(df.head(5))
#print(df.columns)
#
# pprint(df_qa_res.loc[11864,'labels'])
# pprint("ieoiwoew-----------------------------==================")
# pprint(df_qa_res.loc[11864,'response'])
# df_response = df_qa_res[['source_id', 'response']]
# df_response.head(2)

In [14]:
df_res = pd.read_json('../data/ragtruth/raw/response.jsonl', lines=True)
df_res.to_csv("../data/ragtruth/response_dataset.csv", index=False)
print(df_res.columns)
df_res = df_res.drop(columns=["id"]).copy()
df_res.head(6)
df_res["hallucination_label"] = df_res["labels"].apply(
    lambda x: "GROUNDED" if len(x) == 0 else "HALLUCINATED")
df_res = df_res.rename(columns={"labels":"hallucination_span"})
df_res.head(6)


Index(['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality',
       'response'],
      dtype='object')


,source_id,model,temperature,hallucination_span,split,quality,response,hallucination_label
0,15592,gpt-4-0613,0.7,[],train,good,The Anne Frank House has revealed that Anne Fr...,GROUNDED
1,15592,gpt-3.5-turbo-0613,0.7,[],train,good,New research released by the Anne Frank House ...,GROUNDED
2,15592,mistral-7B-instruct,0.7,"[{'start': 636, 'end': 653, 'text': 'February ...",train,good,New research conducted by the Anne Frank House...,HALLUCINATED
3,15592,llama-2-7b-chat,0.7,"[{'start': 693, 'end': 754, 'text': 'Anne's di...",train,good,Here is a summary of the article in 115 words ...,HALLUCINATED
4,15592,llama-2-13b-chat,0.7,[],train,good,"Seventy years ago, Anne Frank died of typhus i...",GROUNDED
5,15592,llama-2-70b-chat,0.7,"[{'start': 348, 'end': 388, 'text': 'sisters d...",train,good,New research suggests that Anne Frank and her ...,HALLUCINATED


## final df

In [8]:
df_final = df_ques_source.merge(df_res, on='source_id', how='inner')
print(df_final.shape)
print(df_final.columns)
df_final.head(5)


(5934, 10)
Index(['question', 'passages', 'source_id', 'model', 'temperature',
       'hallucination_span', 'split', 'quality', 'response',
       'hallucination_label'],
      dtype='object')


,question,passages,source_id,model,temperature,hallucination_span,split,quality,response,hallucination_label
0,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,gpt-4-0613,0.70,[],train,good,The phone numbers for several butcher shops ar...,GROUNDED
1,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,gpt-3.5-turbo-0613,0.70,[],train,good,Butcher Shop - Hayward phone number: (510) 889...,GROUNDED
2,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,mistral-7B-instruct,0.70,"[{'start': 102, 'end': 214, 'text': 'However, ...",train,good,"Based on the given passages, there are several...",HALLUCINATED
3,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,llama-2-7b-chat,0.85,[],train,good,"Based on the provided passages, the butcher sh...",GROUNDED
4,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,llama-2-13b-chat,0.70,[],train,good,"Based on the given passages, the butcher shop ...",GROUNDED


In [9]:
df_final["hallucination_label"].value_counts()
print(df_final["split"].value_counts())

print("train",df_final[df_final["split"] == "train" ]["hallucination_label"].value_counts())
print("test", df_final[df_final["split"] == "test" ]["hallucination_label"].value_counts())

split
train    5034
test      900
Name: count, dtype: int64
train hallucination_label
GROUNDED        3470
HALLUCINATED    1564
Name: count, dtype: int64
test hallucination_label
GROUNDED        740
HALLUCINATED    160
Name: count, dtype: int64


## Original split
Train: 5,034 responses- 3,470 grounded
- 1,564 hallucinated

Test: 900 responses- 740 grounded
- 160 hallucinated

No source overlap between train and test

In [17]:
overlap = df_final[df_final["split"] == "train" ]["source_id"].isin(df_final[df_final["split"] == "test" ]["source_id"])
df_final.loc[df_final["split"] == "train", "source_id"]
print(overlap[overlap])
print(overlap.sum())
df_final.head(5)
df_final.loc[9, "hallucination_span"]

Series([], Name: source_id, dtype: bool)
0


[]

In [11]:
df_hhem = df_final.drop(columns=["model","temperature","hallucination_span","quality"])
df_hhem_validate = df_hhem[df_hhem["split"]== "train"]
df_hhem_test = df_hhem[df_hhem["split"]== "test"]
print(df_hhem.shape)
print(df_hhem_validate.shape)
print(df_hhem_test.shape)
df_hhem_validate.head(2)

(5934, 6)
(5034, 6)
(900, 6)


,question,passages,source_id,split,response,hallucination_label
0,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,train,The phone numbers for several butcher shops ar...,GROUNDED
1,butcher shop phone number,"passage 1:Butcher Shop - Hayward 826 B Street,...",14292,train,Butcher Shop - Hayward phone number: (510) 889...,GROUNDED


In [12]:
passage = df_hhem_validate.loc[
    df_hhem_validate["source_id"] == 14292,
    "passages"
].iloc[0]

print(passage)

response = df_hhem_validate.loc[
    df_hhem_validate["source_id"] == 14292,
    "response"
].iloc[2]

print(response)


passage 1:Butcher Shop - Hayward 826 B Street, Hayward CA 94541 Phone Number: (510) 889-8690

passage 2:Butcher Shop - Lakefield 212 Main Street, Lakefield MN 56150 Phone Number: (507) 662-6281

passage 3:Some More Phone Numbers Related To The Butcher Shop The Local Butcher Shop phone number : (510) 845-6328 Boones Butcher Shop phone number : (502) 348-3668 Ye Ole Butcher Shop phone number : (972) 423-1848 J Ms Butcher Shop phone number : (865) 483-9228


Based on the given passages, there are several different phone numbers associated with butcher shops. However, none of them are for a specific location named "bear" or have any indication of being related to bears. Therefore, it is unable to determine if there is a butcher shop phone number specifically for "bear".


In [13]:
print("validate \n",df_hhem_validate[["passages", "response", "hallucination_label"]].isnull().sum())

print("test\n", df_hhem_test[["passages", "response", "hallucination_label"]].isnull().sum())

validate 
 passages               0
response               0
hallucination_label    0
dtype: int64
test
 passages               0
response               0
hallucination_label    0
dtype: int64
